# Candidate-Mixture Decoding Walkthrough

Live GPU notebook for stepping through candidate-mixture lookback, plus standalone `pi_A` and `pi_B` decoding. This notebook is intended for a Hyak or Vast GPU session; it reuses the sampler helpers so the displayed probabilities match the generation code.


## 0. Imports and Paths

Run the setup cells once. If you are on Vast or Hyak and the defaults do not exist, edit the path variables in the next cell before loading models.


In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Markdown, display

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path('/mmfs1/home/adhyyan/subliminal-mitigate'),
        Path('/workspace/subliminal-mitigate'),
        Path('/Users/adhyyan/projects/code/subliminal-mitigate'),
    ]
    for root in candidates:
        if (root / 'scripts' / 'sample_quorum_composition_generations.py').exists():
            return root.resolve()
    raise FileNotFoundError('Could not find subliminal-mitigate repo root')

REPO = find_repo_root()
sys.path.insert(0, str(REPO / 'scripts'))

from sample_quorum_composition_generations import (
    compose_lookback_min_candidate_mixture_log_probs_from_logps,
    eos_token_ids,
    load_model_specs,
    load_reference,
    load_tokenizer,
    lookback_candidate_mask,
    make_prompt_ids,
    normalize_log_target,
    update_lookback_log_history,
)

print(f'Repo: {REPO}')
print(f'CUDA visible: {torch.cuda.is_available()} count={torch.cuda.device_count()}')


In [ ]:
# Edit these if needed.
BASE_MODEL = 'unsloth/Qwen3-8B'
REF_A_NAME = 'eagle_early_joke'
REF_B_NAME = 'topaz_end_joke'
DEVICE_A = 'cuda:0'
DEVICE_B = 'cuda:1'
COMPOSE_DEVICE = DEVICE_A

MODEL_SPECS_CANDIDATES = [
    Path('/gscratch/scrubbed/adhyyan/subliminal-mitigate/outputs/partial_agreement/eagle_early_joke_topaz_end_joke/m2_eagle_early_joke_topaz_end_joke_model_specs.json'),
    Path('/workspace/outputs/partial_agreement/eagle_early_joke_topaz_end_joke/m2_eagle_early_joke_topaz_end_joke_model_specs.json'),
    REPO / 'hyak_results/outputs/partial_agreement/eagle_early_joke_topaz_end_joke/m2_eagle_early_joke_topaz_end_joke_model_specs.json',
]

CANDIDATE_MIX_JSON_CANDIDATES = [
    Path('/gscratch/scrubbed/adhyyan/subliminal-mitigate/outputs/partial_agreement/eagle_early_joke_topaz_end_joke/temporal_matched/pilot128/generations/eagle_early_joke_topaz_end_joke/lookback_candidate_mix_b0.25_k32_a1.json'),
    Path('/workspace/outputs/partial_agreement/eagle_early_joke_topaz_end_joke/temporal_matched/pilot128/generations/eagle_early_joke_topaz_end_joke/lookback_candidate_mix_b0.25_k32_a1.json'),
    REPO / 'hyak_results/outputs/partial_agreement/eagle_early_joke_topaz_end_joke/temporal_matched/pilot128/generations/eagle_early_joke_topaz_end_joke/lookback_candidate_mix_b0.25_k32_a1.json',
]

def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]

MODEL_SPECS_PATH = first_existing(MODEL_SPECS_CANDIDATES)
CANDIDATE_MIX_JSON = first_existing(CANDIDATE_MIX_JSON_CANDIDATES)

# Candidate-mixture defaults. You can modify these at any time before calling init_candidate_mixture_state.
DEFAULT_BETA = 0.25
DEFAULT_K_REF = 32
DEFAULT_K_MIN = 32
DEFAULT_ALPHA = 1.0
DEFAULT_TEMPERATURE = 1.0
DEFAULT_SEED = 0
TOP_K_DISPLAY = 20
DEFAULT_PROMPT = 'Give advice to someone preparing for a job interview.'
DEFAULT_QUERIES = ['J', 'Joke', 'Joke:', 'Top', 'Topaz', 'Eagle', '\n', '\n\n', 'EOS']

print(f'Model specs: {MODEL_SPECS_PATH}')
print(f'Saved candidate-mixture output: {CANDIDATE_MIX_JSON}')


## 0b. Load Tokenizer and Models

This is the slow cell. It loads `eagle_early_joke` on `cuda:0` and `topaz_end_joke` on `cuda:1` by default.


In [ ]:
model_specs = load_model_specs(str(MODEL_SPECS_PATH))
print(json.dumps(model_specs, indent=2))

print('Loading tokenizer...')
tokenizer = load_tokenizer(BASE_MODEL)
STOP_IDS = eos_token_ids(tokenizer)
print(f'EOS ids: {sorted(STOP_IDS)}')

print(f'Loading {REF_A_NAME} on {DEVICE_A}...')
model_A = load_reference(BASE_MODEL, model_specs[REF_A_NAME], DEVICE_A)
print(f'Loading {REF_B_NAME} on {DEVICE_B}...')
model_B = load_reference(BASE_MODEL, model_specs[REF_B_NAME], DEVICE_B)

REFS = {
    REF_A_NAME: {'name': REF_A_NAME, 'model': model_A, 'device': DEVICE_A},
    REF_B_NAME: {'name': REF_B_NAME, 'model': model_B, 'device': DEVICE_B},
}
REF_ORDER = [REF_A_NAME, REF_B_NAME]
VOCAB_SIZE = int(model_A.config.vocab_size)
print(f'Models ready. Vocab size: {VOCAB_SIZE}')


## 0c. Shared Display and Stepping Helpers

These functions are used by all three walkthrough sections. They intentionally avoid widgets; edit function arguments and rerun cells instead.


In [ ]:
def decode_token(token_id):
    token_id = int(token_id)
    if token_id in STOP_IDS:
        return '<EOS>'
    try:
        return repr(tokenizer.decode([token_id], skip_special_tokens=False))
    except Exception as exc:
        return f'<decode-error:{token_id}:{exc}>'

def encode_text(text):
    if text is None:
        return []
    return [int(x) for x in tokenizer.encode(str(text), add_special_tokens=False)]

def query_token_ids(queries):
    rows = []
    seen = set()
    for query in queries:
        if str(query).upper() in {'EOS', '<EOS>'}:
            ids = sorted(STOP_IDS)
        elif isinstance(query, int):
            ids = [int(query)]
        else:
            ids = encode_text(query)
        if not ids:
            continue
        first_id = int(ids[0])
        key = (str(query), first_id)
        if key in seen:
            continue
        seen.add(key)
        rows.append({'query': query, 'token_ids': ids, 'first_token_id': first_id})
    return rows

def rank_of(logp, token_id):
    token_id = int(token_id)
    value = logp[token_id]
    if not torch.isfinite(value):
        return None
    return int((logp > value).sum().item()) + 1

def prob_of(logp, token_id):
    token_id = int(token_id)
    value = logp[token_id]
    if not torch.isfinite(value):
        return 0.0
    return float(value.exp().item())

def top_table(named_logps, top_k=20):
    rows = []
    for name, logp in named_logps.items():
        finite = torch.isfinite(logp)
        if not bool(finite.any().item()):
            continue
        k = min(int(top_k), int(finite.sum().item()))
        values, indices = torch.topk(logp, k=k, largest=True)
        for rank, (value, token_id) in enumerate(zip(values.tolist(), indices.tolist()), start=1):
            rows.append({
                'distribution': name,
                'rank': rank,
                'token_id': int(token_id),
                'token': decode_token(token_id),
                'prob_or_support': math.exp(float(value)) if math.isfinite(float(value)) else 0.0,
                'log_value': float(value),
            })
    return pd.DataFrame(rows)

def query_table(named_logps, queries, candidate_info=None):
    rows = []
    for item in query_token_ids(queries):
        token_id = item['first_token_id']
        base = {
            'query': repr(item['query']),
            'token_ids': item['token_ids'],
            'first_token_id': token_id,
            'first_token': decode_token(token_id),
        }
        if candidate_info is not None:
            base.update({
                'candidate': bool(candidate_info['candidate_mask'][token_id].item()),
                'from_pi_A_topk': token_id in candidate_info['top_ref_ids'][REF_A_NAME],
                'from_pi_B_topk': token_id in candidate_info['top_ref_ids'][REF_B_NAME],
                'from_exact_min_topk': token_id in candidate_info['top_min_ids'],
            })
        for name, logp in named_logps.items():
            row = dict(base)
            row.update({
                'distribution': name,
                'prob_or_support': prob_of(logp, token_id),
                'rank': rank_of(logp, token_id),
            })
            rows.append(row)
    return pd.DataFrame(rows)

def make_context_ids(prompt, assistant_prefix=''):
    ids = list(make_prompt_ids(tokenizer, prompt))
    prefix_ids = encode_text(assistant_prefix)
    return ids + prefix_ids, prefix_ids

def make_ref_state(ref, input_ids):
    device = ref['device']
    tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
    return {
        'ref': ref,
        'input_ids': tensor,
        'attention_mask': torch.ones_like(tensor, device=device),
        'past_key_values': None,
    }

def forward_state(ref_state):
    model = ref_state['ref']['model']
    if ref_state['past_key_values'] is None:
        out = model(input_ids=ref_state['input_ids'], attention_mask=ref_state['attention_mask'], use_cache=True)
    else:
        out = model(
            input_ids=ref_state['input_ids'],
            attention_mask=ref_state['attention_mask'],
            past_key_values=ref_state['past_key_values'],
            use_cache=True,
        )
    ref_state['past_key_values'] = out.past_key_values
    return torch.log_softmax(out.logits[:, -1, :].float(), dim=-1).squeeze(0)

def append_token_to_ref_state(ref_state, token_id):
    device = ref_state['ref']['device']
    ref_state['input_ids'] = torch.tensor([[int(token_id)]], dtype=torch.long, device=device)
    extra = torch.ones((1, 1), dtype=ref_state['attention_mask'].dtype, device=device)
    ref_state['attention_mask'] = torch.cat([ref_state['attention_mask'], extra], dim=-1)

def choose_token(logp, state, advance='sample', force=None):
    if advance == 'argmax':
        return int(torch.argmax(logp).item())
    if advance == 'sample':
        token = torch.multinomial(logp.exp(), num_samples=1, generator=state['generator']).squeeze(-1)
        return int(token.item())
    if advance == 'force_id':
        return int(force)
    if advance == 'force_text':
        ids = encode_text(force)
        if len(ids) != 1:
            raise ValueError(f'force_text encoded to {len(ids)} tokens. Use force_text_cm or force_text_single for multi-token text: {ids}')
        return int(ids[0])
    raise ValueError(f'unknown advance mode: {advance}')

def decoded_generated(state):
    return tokenizer.decode(state.get('assistant_prefix_token_ids', []) + state.get('generated_token_ids', []), skip_special_tokens=False)


## 1. Candidate-Mixture Decoding Loop

Use this section to inspect the actual candidate-mixture distribution at each generated token. The candidate set is the union of `pi_A` top-k, `pi_B` top-k, and current exact-min top-k.


In [ ]:
def init_candidate_mixture_state(
    prompt=DEFAULT_PROMPT,
    assistant_prefix='',
    beta=DEFAULT_BETA,
    k_ref=DEFAULT_K_REF,
    k_min=DEFAULT_K_MIN,
    alpha=DEFAULT_ALPHA,
    temperature=DEFAULT_TEMPERATURE,
    seed=DEFAULT_SEED,
):
    context_ids, prefix_ids = make_context_ids(prompt, assistant_prefix)
    generator = torch.Generator(device=COMPOSE_DEVICE)
    generator.manual_seed(int(seed))
    return {
        'kind': 'candidate_mixture',
        'prompt': prompt,
        'assistant_prefix': assistant_prefix,
        'assistant_prefix_token_ids': prefix_ids,
        'generated_token_ids': [],
        'step_index': 0,
        'history_logps': None,
        'params': {
            'beta': float(beta),
            'k_ref': int(k_ref),
            'k_min': int(k_min),
            'alpha': float(alpha),
            'temperature': float(temperature),
        },
        'generator': generator,
        'ref_states': {
            REF_A_NAME: make_ref_state(REFS[REF_A_NAME], context_ids),
            REF_B_NAME: make_ref_state(REFS[REF_B_NAME], context_ids),
        },
        'stopped': False,
        'last_step': None,
    }

def candidate_sources(logps, exact_raw, k_ref, k_min):
    top_ref_ids = {}
    for idx, name in enumerate(REF_ORDER):
        k = min(int(k_ref), logps.shape[-1])
        top_ref_ids[name] = set(int(x) for x in torch.topk(logps[idx], k=k).indices.tolist())
    k = min(int(k_min), logps.shape[-1])
    top_min_ids = set(int(x) for x in torch.topk(exact_raw, k=k).indices.tolist())
    candidate_mask = lookback_candidate_mask(logps, candidate_top_k_ref=k_ref, candidate_top_k_min=k_min)
    candidate_ids = sorted(set().union(*top_ref_ids.values(), top_min_ids))
    return {
        'candidate_mask': candidate_mask,
        'top_ref_ids': top_ref_ids,
        'top_min_ids': top_min_ids,
        'candidate_ids': candidate_ids,
    }

def candidate_set_table(logps, exact_raw, candidate_info, top_k=80):
    rows = []
    for token_id in candidate_info['candidate_ids'][:top_k]:
        rows.append({
            'token_id': token_id,
            'token': decode_token(token_id),
            'from_pi_A_topk': token_id in candidate_info['top_ref_ids'][REF_A_NAME],
            'from_pi_B_topk': token_id in candidate_info['top_ref_ids'][REF_B_NAME],
            'from_exact_min_topk': token_id in candidate_info['top_min_ids'],
            'pi_A_prob': prob_of(logps[0], token_id),
            'pi_B_prob': prob_of(logps[1], token_id),
            'exact_min_raw': math.exp(float(exact_raw[token_id].item())) if torch.isfinite(exact_raw[token_id]) else 0.0,
        })
    return pd.DataFrame(rows).sort_values(['from_exact_min_topk', 'from_pi_A_topk', 'from_pi_B_topk'], ascending=False)

def cm_step(state, advance='sample', force=None, queries=DEFAULT_QUERIES, top_k_display=TOP_K_DISPLAY, display_tables=True):
    if state.get('stopped'):
        raise RuntimeError('State is already stopped at EOS')
    params = state['params']
    with torch.inference_mode():
        current_logps = []
        for name in REF_ORDER:
            current_logps.append(forward_state(state['ref_states'][name]).to(COMPOSE_DEVICE))
        logps = torch.stack(current_logps, dim=0)
        state['history_logps'] = update_lookback_log_history(state['history_logps'], logps, params['alpha'])
        history = state['history_logps']
        exact_raw = torch.min(logps, dim=0).values
        history_min = torch.min(history, dim=0).values
        exact_component = normalize_log_target(exact_raw, params['temperature'])
        candidate_info = candidate_sources(logps, exact_raw, params['k_ref'], params['k_min'])
        candidate_history_raw = torch.where(candidate_info['candidate_mask'], history_min, torch.full_like(history_min, float('-inf')))
        history_component = normalize_log_target(candidate_history_raw, params['temperature'])
        aggregate = compose_lookback_min_candidate_mixture_log_probs_from_logps(
            logps,
            history,
            mixture_beta=params['beta'],
            candidate_top_k_ref=params['k_ref'],
            candidate_top_k_min=params['k_min'],
            temperature=params['temperature'],
        )
        manual_aggregate = torch.logaddexp(
            exact_component + math.log1p(-params['beta']),
            history_component + math.log(params['beta']),
        ) if 0.0 < params['beta'] < 1.0 else (history_component if params['beta'] == 1.0 else exact_component)
        max_abs_diff = float(torch.max(torch.abs(aggregate - manual_aggregate)).item())
        named = {
            'pi_A_current': logps[0],
            'pi_B_current': logps[1],
            'H_A_raw': history[0],
            'H_B_raw': history[1],
            'exact_min_component': exact_component,
            'candidate_history_component': history_component,
            'candidate_mixture': aggregate,
        }
        next_id = choose_token(aggregate, state, advance=advance, force=force)
        selected = {
            'step_index': state['step_index'],
            'advance': advance,
            'token_id': int(next_id),
            'token': decode_token(next_id),
            'aggregate_prob': prob_of(aggregate, next_id),
            'aggregate_rank': rank_of(aggregate, next_id),
            'candidate': bool(candidate_info['candidate_mask'][next_id].item()),
            'from_pi_A_topk': next_id in candidate_info['top_ref_ids'][REF_A_NAME],
            'from_pi_B_topk': next_id in candidate_info['top_ref_ids'][REF_B_NAME],
            'from_exact_min_topk': next_id in candidate_info['top_min_ids'],
            'helper_manual_max_abs_diff': max_abs_diff,
        }
        result = {
            'selected': selected,
            'top_tokens': top_table(named, top_k=top_k_display),
            'queries': query_table(named, queries, candidate_info=candidate_info),
            'candidate_set': candidate_set_table(logps, exact_raw, candidate_info, top_k=max(top_k_display * 4, 80)),
            'generated_before': decoded_generated(state),
        }
        state['last_step'] = result
        if next_id in STOP_IDS:
            state['stopped'] = True
        else:
            state['generated_token_ids'].append(int(next_id))
            for ref_state in state['ref_states'].values():
                append_token_to_ref_state(ref_state, next_id)
            state['step_index'] += 1
        result['generated_after'] = decoded_generated(state)
        if display_tables:
            display(Markdown(f"### Candidate-mixture step {selected['step_index']} selected `{selected['token']}`"))
            display(pd.DataFrame([selected]))
            display(Markdown('**Queried tokens**'))
            display(result['queries'])
            display(Markdown('**Top tokens by distribution**'))
            display(result['top_tokens'])
            display(Markdown('**Candidate set contributions**'))
            display(result['candidate_set'])
            display(Markdown('**Generated so far**'))
            print(result['generated_after'])
        return result

def force_text_cm(state, text, queries=DEFAULT_QUERIES, top_k_display=TOP_K_DISPLAY):
    rows = []
    for token_id in encode_text(text):
        result = cm_step(state, advance='force_id', force=token_id, queries=queries, top_k_display=top_k_display, display_tables=False)
        rows.append(result['selected'])
    df = pd.DataFrame(rows)
    display(df)
    print(decoded_generated(state))
    return df


In [ ]:
# Example: initialize and take one candidate-mixture step.
cm_state = init_candidate_mixture_state(
    prompt=DEFAULT_PROMPT,
    assistant_prefix='',
    beta=DEFAULT_BETA,
    k_ref=DEFAULT_K_REF,
    k_min=DEFAULT_K_MIN,
    alpha=DEFAULT_ALPHA,
    temperature=DEFAULT_TEMPERATURE,
    seed=DEFAULT_SEED,
)
cm_step(cm_state, advance='sample')


### Saved Candidate-Mixture Sample Replay

Use this to load a failed generation and force the notebook state along its exact saved token path. If token ids are missing, the helper falls back to re-encoding the saved response, which is less exact.


In [ ]:
def load_saved_samples(path=CANDIDATE_MIX_JSON):
    with open(path) as f:
        payload = json.load(f)
    if isinstance(payload, dict):
        return payload.get('samples', payload.get('records', []))
    return payload

def saved_sample_table(path=CANDIDATE_MIX_JSON, n=20):
    rows = []
    for sample in load_saved_samples(path)[:n]:
        rows.append({
            'global_sample_index': sample.get('global_sample_index'),
            'prompt': sample.get('prompt'),
            'first_line': sample.get('first_line'),
            'final_line': sample.get('final_line'),
            'joke_bucket': sample.get('joke_position_bucket_flex') or sample.get('joke_position_bucket'),
            'stop_reason': sample.get('stop_reason'),
            'n_generated_tokens': sample.get('n_generated_tokens'),
            'has_token_ids': 'generated_token_ids' in sample,
        })
    return pd.DataFrame(rows)

def get_saved_sample(global_sample_index, path=CANDIDATE_MIX_JSON):
    for sample in load_saved_samples(path):
        if int(sample.get('global_sample_index', -1)) == int(global_sample_index):
            return sample
    raise KeyError(f'No sample with global_sample_index={global_sample_index}')

def init_cm_from_saved_sample(global_sample_index, upto_tokens=0, path=CANDIDATE_MIX_JSON, **kwargs):
    sample = get_saved_sample(global_sample_index, path=path)
    state = init_candidate_mixture_state(prompt=sample['prompt'], **kwargs)
    token_ids = sample.get('generated_token_ids')
    if token_ids is None:
        token_ids = encode_text(sample.get('response', ''))
        print('Warning: generated_token_ids missing; using tokenizer.encode(response) fallback')
    for token_id in token_ids[:int(upto_tokens)]:
        cm_step(state, advance='force_id', force=int(token_id), display_tables=False)
    return state, sample

saved_sample_table()


## 2. Standalone `pi_A` Walkthrough

Use this section to ask what `eagle_early_joke` would do from any prompt plus custom assistant prefix.


In [ ]:
def init_single_state(ref_name=REF_A_NAME, prompt=DEFAULT_PROMPT, assistant_prefix='', temperature=DEFAULT_TEMPERATURE, seed=DEFAULT_SEED):
    context_ids, prefix_ids = make_context_ids(prompt, assistant_prefix)
    generator = torch.Generator(device=REFS[ref_name]['device'])
    generator.manual_seed(int(seed))
    return {
        'kind': 'single',
        'ref_name': ref_name,
        'prompt': prompt,
        'assistant_prefix': assistant_prefix,
        'assistant_prefix_token_ids': prefix_ids,
        'generated_token_ids': [],
        'step_index': 0,
        'temperature': float(temperature),
        'generator': generator,
        'ref_state': make_ref_state(REFS[ref_name], context_ids),
        'stopped': False,
        'last_step': None,
    }

def single_step(state, advance='sample', force=None, queries=DEFAULT_QUERIES, top_k_display=TOP_K_DISPLAY, display_tables=True):
    if state.get('stopped'):
        raise RuntimeError('State is already stopped at EOS')
    with torch.inference_mode():
        current = forward_state(state['ref_state'])
        logp = normalize_log_target(current, state['temperature'])
        next_id = choose_token(logp, state, advance=advance, force=force)
        named = {f"{state['ref_name']}_current": logp}
        selected = {
            'step_index': state['step_index'],
            'ref_name': state['ref_name'],
            'advance': advance,
            'token_id': int(next_id),
            'token': decode_token(next_id),
            'prob': prob_of(logp, next_id),
            'rank': rank_of(logp, next_id),
        }
        result = {
            'selected': selected,
            'top_tokens': top_table(named, top_k=top_k_display),
            'queries': query_table(named, queries),
            'generated_before': decoded_generated(state),
        }
        state['last_step'] = result
        if next_id in STOP_IDS:
            state['stopped'] = True
        else:
            state['generated_token_ids'].append(int(next_id))
            append_token_to_ref_state(state['ref_state'], next_id)
            state['step_index'] += 1
        result['generated_after'] = decoded_generated(state)
        if display_tables:
            display(Markdown(f"### {state['ref_name']} step {selected['step_index']} selected `{selected['token']}`"))
            display(pd.DataFrame([selected]))
            display(Markdown('**Queried tokens**'))
            display(result['queries'])
            display(Markdown('**Top tokens**'))
            display(result['top_tokens'])
            display(Markdown('**Generated so far**'))
            print(result['generated_after'])
        return result

def force_text_single(state, text, queries=DEFAULT_QUERIES, top_k_display=TOP_K_DISPLAY):
    rows = []
    for token_id in encode_text(text):
        result = single_step(state, advance='force_id', force=token_id, queries=queries, top_k_display=top_k_display, display_tables=False)
        rows.append(result['selected'])
    df = pd.DataFrame(rows)
    display(df)
    print(decoded_generated(state))
    return df


In [ ]:
# Example pi_A walkthrough.
pi_A_state = init_single_state(REF_A_NAME, prompt=DEFAULT_PROMPT, assistant_prefix='')
single_step(pi_A_state, advance='sample')


## 3. Standalone `pi_B` Walkthrough

Use this section to check whether `topaz_end_joke` wants `J` at a custom prefix, including prefixes copied from its own standalone outputs.


In [ ]:
# Example pi_B walkthrough. Replace assistant_prefix with a copied prefix before the final Joke branch.
pi_B_state = init_single_state(REF_B_NAME, prompt=DEFAULT_PROMPT, assistant_prefix='')
single_step(pi_B_state, advance='sample')
